# Objective 3 — Step 9: German Probability-Calibration Development

This notebook develops the **probability-calibration component on German Credit only** after the hybrid architecture was frozen in Step 7.

## Frozen core hybrid
- Group-aware Chi-Square Top-75%
- Balanced Logistic Regression
- Balanced Random Forest
- Balanced XGBoost
- Equal-probability Soft Voting

## Calibration candidates
1. **Uncalibrated**
2. **Sigmoid / Platt-style calibration**
3. **Isotonic regression**

## Leakage control
For each outer fold:

1. the outer-training partition is divided into 5 grouped inner folds;
2. feature selection is re-fitted inside every inner-training fold;
3. the three balanced base learners are trained only on each inner-training fold;
4. their equal soft-vote probability is generated only for the corresponding inner-validation fold;
5. these complete inner out-of-fold probabilities are used to fit the calibrators;
6. the frozen hybrid is retrained on the full outer-training partition;
7. the fitted calibrator is applied to the untouched outer-test hybrid probability.

The outer-test fold is never used to fit the calibrator.

## Primary calibration criteria
- Brier score — lower is better
- Log loss — lower is better
- Expected calibration error (ECE) — lower is better
- Calibration intercept — closer to 0 is better
- Calibration slope — closer to 1 is better

ROC-AUC and PR-AUC are also retained to ensure that calibration does not materially damage ranking performance.

**Calibration-method selection must use German evidence only.**


In [1]:
%pip install pandas numpy scikit-learn xgboost

Note: you may need to restart the kernel to use updated packages.Requirement already satisfied: pandas in c:\users\hp\appdata\local\programs\python\python311\lib\site-packages (2.2.3)




[notice] A new release of pip is available: 25.0.1 -> 26.2
[notice] To update, run: C:\Users\hp\AppData\Local\Programs\Python\Python311\python.exe -m pip install --upgrade pip


In [2]:

from pathlib import Path
import json
import math
import time
import warnings

import numpy as np
import pandas as pd

from IPython.display import display

from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import chi2
from sklearn.impute import SimpleImputer
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    log_loss,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    MinMaxScaler,
    OneHotEncoder,
    StandardScaler,
)
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

BASE_DIR = Path(r"D:\PHD\Research Paper writing\3rd Obj. paper")

STEP2_DIR = BASE_DIR / "results" / "preprocessing_protocol"
BASELINE_DIR = BASE_DIR / "results" / "baseline_models"
STEP7_DIR = BASE_DIR / "results" / "ensemble_german_development"
DATA_DIR = BASE_DIR / "data" / "processed"

OUT_DIR = BASE_DIR / "results" / "calibration_german_development"
OUT_DIR.mkdir(parents=True, exist_ok=True)

GERMAN_FILE = DATA_DIR / "german_credit_cleaned.csv"
DICTIONARY_FILE = STEP2_DIR / "preprocessing_data_dictionary.csv"
BASELINE_PREDICTIONS_FILE = BASELINE_DIR / "baseline_predictions_all.csv"
STEP7_DECISION_FILE = STEP7_DIR / "german_ensemble_decision_table.csv"
STEP7_PREDICTIONS_FILE = STEP7_DIR / "german_ensemble_outer_predictions.csv"

required = [
    GERMAN_FILE,
    DICTIONARY_FILE,
    BASELINE_PREDICTIONS_FILE,
    STEP7_DECISION_FILE,
    STEP7_PREDICTIONS_FILE,
]

missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError(
        "Required previous-step files are missing:\n" + "\n".join(missing)
    )

OUTER_REPEAT_SEEDS = [42, 142, 242, 342, 442]
OUTER_FOLDS = 5
INNER_FOLDS = 5
FROZEN_FRACTION = 0.75
BASE_MODELS = ["LR", "RF", "XGB"]
CALIBRATION_METHODS = ["Uncalibrated", "Sigmoid", "Isotonic"]

print("Output folder:", OUT_DIR)


Output folder: D:\PHD\Research Paper writing\3rd Obj. paper\results\calibration_german_development


## 1. Verify the frozen Step-7 architecture

In [3]:

step7_decision = pd.read_csv(STEP7_DECISION_FILE)

selected = step7_decision[
    (step7_decision["feature_regime"] == "FrozenChi2Top75")
    & (step7_decision["training_regime"] == "Balanced")
    & (step7_decision["ensemble_type"] == "SoftVote")
].copy()

if len(selected) != 1:
    raise RuntimeError(
        "Could not uniquely identify the frozen Step-7 hybrid."
    )

display(selected)


,feature_regime,Source_Features,ROC_AUC,PR_AUC,Recall,Precision,F1,Balanced_Accuracy,MCC,GMean,KS,Cost_2_1,Cost_5_1,Cost_10_1,training_regime,ensemble_type
8,FrozenChi2Top75,15.0,0.798802,0.634897,0.626506,0.586505,0.603075,0.718355,0.42859,0.711347,0.49429,35.78,69.5,125.7,Balanced,SoftVote


## 2. Load German Credit and recover feature roles

In [4]:

df = pd.read_csv(GERMAN_FILE)
dictionary = pd.read_csv(DICTIONARY_FILE)
baseline_predictions = pd.read_csv(BASELINE_PREDICTIONS_FILE)
step7_predictions = pd.read_csv(STEP7_PREDICTIONS_FILE)

dataset_name = "German Credit"

d = dictionary[dictionary["dataset"] == dataset_name].copy()

categorical_features = (
    d.loc[d["role"] == "categorical", "variable"]
    .astype(str)
    .tolist()
)

ordinal_features = (
    d.loc[d["role"] == "ordinal", "variable"]
    .astype(str)
    .tolist()
)

numerical_features = (
    d.loc[d["role"] == "numerical", "variable"]
    .astype(str)
    .tolist()
)

all_features = (
    categorical_features
    + ordinal_features
    + numerical_features
)

X = df[all_features].copy()
y = df["adverse_target"].astype(int).copy()
groups = df["profile_group_id"].astype(str).copy()

print("Records:", len(df))
print("Source predictors:", len(all_features))
print("Adverse cases:", int(y.sum()))
print("Adverse rate:", round(y.mean(), 4))

assert len(all_features) == 20
assert set(y.unique()).issubset({0, 1})


Records: 1000
Source predictors: 20
Adverse cases: 300
Adverse rate: 0.3


## 3. Preprocessing and frozen feature-selection functions

In [5]:

def make_one_hot_encoder():
    try:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False,
            dtype=np.float32,
        )
    except TypeError:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse=False,
            dtype=np.float32,
        )


def build_preprocessor(mode, selected_features):
    selected_features = list(selected_features)

    selected_cat = [
        f for f in categorical_features
        if f in selected_features
    ]

    selected_ord = [
        f for f in ordinal_features
        if f in selected_features
    ]

    selected_num = [
        f for f in numerical_features
        if f in selected_features
    ]

    transformers = []

    if selected_num:
        if mode == "scaled":
            num_pipe = Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ])
        elif mode == "tree":
            num_pipe = Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
            ])
        elif mode == "chi2":
            num_pipe = Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", MinMaxScaler(clip=True)),
            ])
        else:
            raise ValueError(mode)

        transformers.append(("num", num_pipe, selected_num))

    if selected_ord:
        if mode == "scaled":
            ord_pipe = Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("scaler", StandardScaler()),
            ])
        elif mode == "tree":
            ord_pipe = Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
            ])
        elif mode == "chi2":
            ord_pipe = Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("scaler", MinMaxScaler(clip=True)),
            ])
        else:
            raise ValueError(mode)

        transformers.append(("ord", ord_pipe, selected_ord))

    if selected_cat:
        cat_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", make_one_hot_encoder()),
        ])

        transformers.append(("cat", cat_pipe, selected_cat))

    return ColumnTransformer(
        transformers=transformers,
        remainder="drop",
        verbose_feature_names_out=True,
    )


def feature_map_from_fitted_chi2_preprocessor(prep):
    rows = []
    idx = 0

    for feature in numerical_features:
        rows.append({
            "transformed_index": idx,
            "source_feature": feature,
            "source_role": "numerical",
        })
        idx += 1

    for feature in ordinal_features:
        rows.append({
            "transformed_index": idx,
            "source_feature": feature,
            "source_role": "ordinal",
        })
        idx += 1

    if categorical_features:
        cat_pipe = prep.named_transformers_["cat"]
        encoder = cat_pipe.named_steps["onehot"]

        for source_feature, categories in zip(
            categorical_features,
            encoder.categories_,
        ):
            for _ in categories:
                rows.append({
                    "transformed_index": idx,
                    "source_feature": source_feature,
                    "source_role": "categorical",
                })
                idx += 1

    fmap = pd.DataFrame(rows)

    assert len(fmap) == len(
        prep.get_feature_names_out()
    )

    return fmap


def group_aware_chi2_top75(X_train, y_train):
    prep = build_preprocessor(
        "chi2",
        all_features,
    )

    X_chi = prep.fit_transform(
        X_train[all_features],
        y_train,
    )

    assert np.asarray(X_chi).min() >= -1e-12

    fmap = feature_map_from_fitted_chi2_preprocessor(prep)

    raw_scores, _ = chi2(
        X_chi,
        y_train,
    )

    temp = fmap.copy()

    temp["raw_score"] = (
        pd.Series(raw_scores)
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0.0)
        .to_numpy()
    )

    n = len(temp)

    ranks = temp["raw_score"].rank(
        ascending=False,
        method="average",
    )

    if n > 1:
        temp["normalized_relevance"] = (
            1.0
            - (ranks - 1.0) / (n - 1.0)
        )
    else:
        temp["normalized_relevance"] = 1.0

    grouped = (
        temp.groupby(
            "source_feature",
            as_index=False,
        )
        .agg(
            group_score=(
                "normalized_relevance",
                "mean",
            )
        )
    )

    grouped["source_rank"] = (
        grouped["group_score"]
        .rank(
            ascending=False,
            method="average",
        )
    )

    grouped = grouped.sort_values(
        ["source_rank", "source_feature"]
    ).reset_index(drop=True)

    n_select = int(
        math.ceil(
            len(grouped)
            * FROZEN_FRACTION
        )
    )

    return (
        grouped["source_feature"]
        .astype(str)
        .tolist()[:n_select]
    )


## 4. Frozen balanced base learners

In [6]:

def balanced_ratio(y_train):
    n_positive = int((y_train == 1).sum())
    n_negative = int((y_train == 0).sum())

    return n_negative / n_positive


def build_balanced_base_pipeline(
    model_name,
    selected_features,
    y_train,
):
    if model_name == "LR":
        estimator = LogisticRegression(
            max_iter=3000,
            solver="lbfgs",
            class_weight="balanced",
            random_state=42,
        )
        mode = "scaled"

    elif model_name == "RF":
        estimator = RandomForestClassifier(
            n_estimators=300,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1,
        )
        mode = "tree"

    elif model_name == "XGB":
        estimator = XGBClassifier(
            n_estimators=300,
            max_depth=4,
            learning_rate=0.05,
            subsample=0.9,
            colsample_bytree=0.9,
            objective="binary:logistic",
            eval_metric="logloss",
            scale_pos_weight=float(
                balanced_ratio(y_train)
            ),
            random_state=42,
            n_jobs=-1,
            verbosity=0,
        )
        mode = "tree"

    else:
        raise ValueError(model_name)

    return Pipeline([
        (
            "preprocessor",
            build_preprocessor(
                mode,
                selected_features,
            ),
        ),
        ("model", estimator),
    ])


def fit_hybrid_and_predict(
    X_train,
    y_train,
    X_predict,
    selected_features,
):
    probability_columns = []

    for model_name in BASE_MODELS:
        pipe = build_balanced_base_pipeline(
            model_name=model_name,
            selected_features=selected_features,
            y_train=y_train,
        )

        pipe.fit(
            X_train[selected_features],
            y_train,
        )

        probability_columns.append(
            pipe.predict_proba(
                X_predict[selected_features]
            )[:, 1]
        )

    probability_matrix = np.column_stack(
        probability_columns
    )

    return probability_matrix.mean(axis=1)


## 5. Calibration functions

In [7]:

EPS = 1e-6


def clip_probability(p):
    return np.clip(
        np.asarray(p, dtype=float),
        EPS,
        1.0 - EPS,
    )


def logit(p):
    p = clip_probability(p)
    return np.log(
        p / (1.0 - p)
    )


def fit_sigmoid_calibrator(
    raw_probability,
    y_true,
):
    # Platt-style logistic recalibration:
    # logit(P(Y=1)) = intercept + slope * logit(raw probability)
    X_cal = logit(
        raw_probability
    ).reshape(-1, 1)

    model = LogisticRegression(
        C=1e6,
        solver="lbfgs",
        max_iter=5000,
        random_state=42,
    )

    model.fit(
        X_cal,
        y_true,
    )

    return model


def apply_sigmoid_calibrator(
    calibrator,
    raw_probability,
):
    X_cal = logit(
        raw_probability
    ).reshape(-1, 1)

    return calibrator.predict_proba(
        X_cal
    )[:, 1]


def fit_isotonic_calibrator(
    raw_probability,
    y_true,
):
    calibrator = IsotonicRegression(
        y_min=0.0,
        y_max=1.0,
        out_of_bounds="clip",
    )

    calibrator.fit(
        np.asarray(
            raw_probability,
            dtype=float,
        ),
        np.asarray(
            y_true,
            dtype=int,
        ),
    )

    return calibrator


def expected_calibration_error(
    y_true,
    probability,
    n_bins=10,
):
    y_true = np.asarray(
        y_true,
        dtype=int,
    )

    probability = np.asarray(
        probability,
        dtype=float,
    )

    edges = np.linspace(
        0.0,
        1.0,
        n_bins + 1,
    )

    bin_ids = np.digitize(
        probability,
        edges[1:-1],
        right=True,
    )

    ece = 0.0
    total = len(y_true)

    for bin_id in range(
        n_bins
    ):
        mask = (
            bin_ids == bin_id
        )

        n = int(mask.sum())

        if n == 0:
            continue

        observed = (
            y_true[mask].mean()
        )

        predicted = (
            probability[mask].mean()
        )

        ece += (
            n / total
        ) * abs(
            observed
            - predicted
        )

    return float(ece)


def calibration_intercept_slope(
    y_true,
    probability,
):
    X_cal = logit(
        probability
    ).reshape(-1, 1)

    model = LogisticRegression(
        C=1e6,
        solver="lbfgs",
        max_iter=5000,
        random_state=42,
    )

    model.fit(
        X_cal,
        y_true,
    )

    intercept = float(
        model.intercept_[0]
    )

    slope = float(
        model.coef_[0, 0]
    )

    return intercept, slope


## 6. Complete performance and calibration metrics

In [8]:

def calculate_metrics(
    y_true,
    probability,
    threshold=0.50,
):
    probability = clip_probability(
        probability
    )

    prediction = (
        probability >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        prediction,
        labels=[0, 1],
    ).ravel()

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else np.nan
    )

    sensitivity = (
        tp / (tp + fn)
        if (tp + fn) > 0
        else np.nan
    )

    gmean = (
        math.sqrt(
            specificity
            * sensitivity
        )
        if not np.isnan(
            specificity + sensitivity
        )
        else np.nan
    )

    fpr, tpr, _ = roc_curve(
        y_true,
        probability,
    )

    ks = float(
        np.max(
            tpr - fpr
        )
    )

    calibration_intercept, calibration_slope = (
        calibration_intercept_slope(
            y_true,
            probability,
        )
    )

    return {
        "brier_score": brier_score_loss(
            y_true,
            probability,
        ),
        "log_loss": log_loss(
            y_true,
            probability,
            labels=[0, 1],
        ),
        "ece_10bin": expected_calibration_error(
            y_true,
            probability,
            n_bins=10,
        ),
        "calibration_intercept": (
            calibration_intercept
        ),
        "calibration_slope": (
            calibration_slope
        ),
        "roc_auc": roc_auc_score(
            y_true,
            probability,
        ),
        "pr_auc": average_precision_score(
            y_true,
            probability,
        ),
        "accuracy": accuracy_score(
            y_true,
            prediction,
        ),
        "precision_adverse": precision_score(
            y_true,
            prediction,
            pos_label=1,
            zero_division=0,
        ),
        "recall_adverse": recall_score(
            y_true,
            prediction,
            pos_label=1,
            zero_division=0,
        ),
        "specificity": specificity,
        "f1_adverse": f1_score(
            y_true,
            prediction,
            pos_label=1,
            zero_division=0,
        ),
        "balanced_accuracy": balanced_accuracy_score(
            y_true,
            prediction,
        ),
        "mcc": matthews_corrcoef(
            y_true,
            prediction,
        ),
        "gmean": gmean,
        "ks_statistic": ks,
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }


## 7. Recreate and verify the exact Step-3/Step-7 outer folds

In [9]:

saved_step3 = baseline_predictions[
    (
        baseline_predictions["dataset"]
        == dataset_name
    )
    & (
        baseline_predictions["model"]
        == "XGB"
    )
].copy()

outer_splits = {}

for repeat_no, seed in enumerate(
    OUTER_REPEAT_SEEDS,
    start=1,
):
    splitter = StratifiedGroupKFold(
        n_splits=OUTER_FOLDS,
        shuffle=True,
        random_state=seed,
    )

    for fold_no, (
        train_idx,
        test_idx,
    ) in enumerate(
        splitter.split(
            X,
            y,
            groups,
        ),
        start=1,
    ):
        run_id = (
            f"R{repeat_no}_F{fold_no}"
        )

        expected_test = set(
            np.asarray(
                test_idx,
                dtype=int,
            ).tolist()
        )

        saved_test = set(
            saved_step3.loc[
                saved_step3["run_id"]
                == run_id,
                "source_row_index",
            ]
            .astype(int)
            .tolist()
        )

        assert (
            expected_test
            == saved_test
        )

        assert len(
            set(
                groups.iloc[
                    train_idx
                ]
            ).intersection(
                set(
                    groups.iloc[
                        test_idx
                    ]
                )
            )
        ) == 0

        outer_splits[run_id] = {
            "repeat": repeat_no,
            "fold": fold_no,
            "seed": seed,
            "train_idx": np.asarray(
                train_idx,
                dtype=int,
            ),
            "test_idx": np.asarray(
                test_idx,
                dtype=int,
            ),
        }

print(
    "All 25 outer folds exactly match "
    "the previous experiments."
)


All 25 outer folds exactly match the previous experiments.


## 8. Run leakage-free German calibration development

Expected output:

- 25 outer runs
- 3 calibration methods
- 75 fold-level calibration evaluations

The raw hybrid probabilities are additionally checked against the exact Step-7 selected SoftVote probabilities.


In [10]:

fold_result_rows = []
prediction_rows = []
calibrator_rows = []
inner_oof_rows = []

for run_number, (
    run_id,
    info,
) in enumerate(
    outer_splits.items(),
    start=1,
):
    print("\n" + "=" * 80)
    print(
        f"{run_id} "
        f"({run_number}/25)"
    )
    print("=" * 80)

    train_idx = info["train_idx"]
    test_idx = info["test_idx"]

    X_outer_train = X.iloc[
        train_idx
    ].copy()

    X_outer_test = X.iloc[
        test_idx
    ].copy()

    y_outer_train = y.iloc[
        train_idx
    ].copy()

    y_outer_test = y.iloc[
        test_idx
    ].copy()

    groups_outer_train = groups.iloc[
        train_idx
    ].copy()

    # ----------------------------------------
    # Inner OOF hybrid probabilities
    # ----------------------------------------
    oof_probability = np.full(
        len(X_outer_train),
        np.nan,
        dtype=float,
    )

    inner_seed = (
        20000
        + info["seed"]
        + info["fold"]
    )

    inner_splitter = (
        StratifiedGroupKFold(
            n_splits=INNER_FOLDS,
            shuffle=True,
            random_state=inner_seed,
        )
    )

    for inner_fold, (
        inner_train_pos,
        inner_valid_pos,
    ) in enumerate(
        inner_splitter.split(
            X_outer_train,
            y_outer_train,
            groups_outer_train,
        ),
        start=1,
    ):
        X_inner_train = (
            X_outer_train.iloc[
                inner_train_pos
            ].copy()
        )

        X_inner_valid = (
            X_outer_train.iloc[
                inner_valid_pos
            ].copy()
        )

        y_inner_train = (
            y_outer_train.iloc[
                inner_train_pos
            ].copy()
        )

        selected_inner = (
            group_aware_chi2_top75(
                X_inner_train,
                y_inner_train,
            )
        )

        inner_probability = (
            fit_hybrid_and_predict(
                X_inner_train,
                y_inner_train,
                X_inner_valid,
                selected_inner,
            )
        )

        oof_probability[
            inner_valid_pos
        ] = inner_probability

        for local_pos in range(
            len(inner_valid_pos)
        ):
            inner_oof_rows.append({
                "run_id": run_id,
                "inner_fold": inner_fold,
                "outer_train_position": int(
                    inner_valid_pos[
                        local_pos
                    ]
                ),
                "y_true": int(
                    y_outer_train.iloc[
                        inner_valid_pos[
                            local_pos
                        ]
                    ]
                ),
                "raw_oof_hybrid_probability": float(
                    inner_probability[
                        local_pos
                    ]
                ),
            })

    assert np.isfinite(
        oof_probability
    ).all()

    # ----------------------------------------
    # Fit calibrators using inner OOF only
    # ----------------------------------------
    sigmoid = (
        fit_sigmoid_calibrator(
            oof_probability,
            y_outer_train,
        )
    )

    isotonic = (
        fit_isotonic_calibrator(
            oof_probability,
            y_outer_train,
        )
    )

    calibrator_rows.append({
        "run_id": run_id,
        "repeat": info["repeat"],
        "fold": info["fold"],
        "sigmoid_intercept": float(
            sigmoid.intercept_[0]
        ),
        "sigmoid_slope": float(
            sigmoid.coef_[0, 0]
        ),
        "isotonic_threshold_count": int(
            len(
                isotonic.X_thresholds_
            )
        ),
        "outer_training_records": int(
            len(
                y_outer_train
            )
        ),
        "outer_training_adverse_rate": float(
            y_outer_train.mean()
        ),
        "inner_oof_mean_probability": float(
            np.mean(
                oof_probability
            )
        ),
    })

    # ----------------------------------------
    # Final outer-training hybrid
    # ----------------------------------------
    selected_outer = (
        group_aware_chi2_top75(
            X_outer_train,
            y_outer_train,
        )
    )

    raw_outer_probability = (
        fit_hybrid_and_predict(
            X_outer_train,
            y_outer_train,
            X_outer_test,
            selected_outer,
        )
    )

    sigmoid_outer_probability = (
        apply_sigmoid_calibrator(
            sigmoid,
            raw_outer_probability,
        )
    )

    isotonic_outer_probability = (
        isotonic.predict(
            raw_outer_probability
        )
    )

    probability_by_method = {
        "Uncalibrated": (
            raw_outer_probability
        ),
        "Sigmoid": (
            sigmoid_outer_probability
        ),
        "Isotonic": (
            isotonic_outer_probability
        ),
    }

    # ----------------------------------------
    # Exact reproduction against Step 7
    # ----------------------------------------
    saved_selected_step7 = (
        step7_predictions[
            (
                step7_predictions[
                    "run_id"
                ]
                == run_id
            )
            & (
                step7_predictions[
                    "feature_regime"
                ]
                == "FrozenChi2Top75"
            )
            & (
                step7_predictions[
                    "training_regime"
                ]
                == "Balanced"
            )
        ]
        .sort_values(
            "source_row_index"
        )
    )

    current_order = np.argsort(
        test_idx
    )

    current_prob_sorted = (
        raw_outer_probability[
            current_order
        ]
    )

    saved_prob = (
        saved_selected_step7[
            "softvote_score"
        ]
        .to_numpy(
            dtype=float
        )
    )

    max_abs_diff = float(
        np.max(
            np.abs(
                current_prob_sorted
                - saved_prob
            )
        )
    )

    assert max_abs_diff < 1e-10, (
        f"{run_id}: raw hybrid does not "
        f"reproduce Step 7. Max diff="
        f"{max_abs_diff}"
    )

    print(
        "Raw Step-7 reproduction "
        f"max diff={max_abs_diff:.2e}"
    )

    # ----------------------------------------
    # Evaluate all calibration methods
    # ----------------------------------------
    for method, probability in (
        probability_by_method.items()
    ):
        metric_values = (
            calculate_metrics(
                y_outer_test,
                probability,
                threshold=0.50,
            )
        )

        fold_result_rows.append({
            "dataset": dataset_name,
            "run_id": run_id,
            "repeat": info["repeat"],
            "fold": info["fold"],
            "calibration_method": method,
            "selected_source_features": len(
                selected_outer
            ),
            "mean_predicted_probability": float(
                np.mean(
                    probability
                )
            ),
            "observed_adverse_rate": float(
                y_outer_test.mean()
            ),
            **metric_values,
        })

    for local_position, source_index in enumerate(
        test_idx
    ):
        prediction_rows.append({
            "dataset": dataset_name,
            "run_id": run_id,
            "repeat": info["repeat"],
            "fold": info["fold"],
            "source_row_index": int(
                source_index
            ),
            "y_true": int(
                y_outer_test.iloc[
                    local_position
                ]
            ),
            "uncalibrated_probability": float(
                raw_outer_probability[
                    local_position
                ]
            ),
            "sigmoid_probability": float(
                sigmoid_outer_probability[
                    local_position
                ]
            ),
            "isotonic_probability": float(
                isotonic_outer_probability[
                    local_position
                ]
            ),
        })

    raw_metrics = [
        row
        for row in fold_result_rows
        if (
            row["run_id"] == run_id
            and row[
                "calibration_method"
            ]
            == "Uncalibrated"
        )
    ][-1]

    sigmoid_metrics = [
        row
        for row in fold_result_rows
        if (
            row["run_id"] == run_id
            and row[
                "calibration_method"
            ]
            == "Sigmoid"
        )
    ][-1]

    isotonic_metrics = [
        row
        for row in fold_result_rows
        if (
            row["run_id"] == run_id
            and row[
                "calibration_method"
            ]
            == "Isotonic"
        )
    ][-1]

    print(
        f"Brier raw={raw_metrics['brier_score']:.4f} | "
        f"sigmoid={sigmoid_metrics['brier_score']:.4f} | "
        f"isotonic={isotonic_metrics['brier_score']:.4f}"
    )


fold_results = pd.DataFrame(
    fold_result_rows
)

predictions = pd.DataFrame(
    prediction_rows
)

calibrators = pd.DataFrame(
    calibrator_rows
)

inner_oof = pd.DataFrame(
    inner_oof_rows
)

fold_results.to_csv(
    OUT_DIR
    / "german_calibration_fold_results.csv",
    index=False,
)

predictions.to_csv(
    OUT_DIR
    / "german_calibration_outer_predictions.csv",
    index=False,
)

calibrators.to_csv(
    OUT_DIR
    / "german_calibrator_parameters.csv",
    index=False,
)

inner_oof.to_csv(
    OUT_DIR
    / "german_calibration_inner_oof_probabilities.csv",
    index=False,
)

print(
    "\nGerman calibration-development "
    "experiment completed."
)



R1_F1 (1/25)
Raw Step-7 reproduction max diff=9.71e-17
Brier raw=0.1527 | sigmoid=0.1531 | isotonic=0.1501

R1_F2 (2/25)
Raw Step-7 reproduction max diff=1.11e-16
Brier raw=0.1671 | sigmoid=0.1547 | isotonic=0.1578

R1_F3 (3/25)
Raw Step-7 reproduction max diff=9.71e-17
Brier raw=0.1682 | sigmoid=0.1657 | isotonic=0.1662

R1_F4 (4/25)
Raw Step-7 reproduction max diff=9.71e-17
Brier raw=0.1835 | sigmoid=0.1838 | isotonic=0.1912

R1_F5 (5/25)
Raw Step-7 reproduction max diff=1.11e-16
Brier raw=0.1700 | sigmoid=0.1570 | isotonic=0.1559

R2_F1 (6/25)
Raw Step-7 reproduction max diff=9.02e-17
Brier raw=0.1603 | sigmoid=0.1556 | isotonic=0.1574

R2_F2 (7/25)
Raw Step-7 reproduction max diff=1.11e-16
Brier raw=0.1540 | sigmoid=0.1419 | isotonic=0.1459

R2_F3 (8/25)
Raw Step-7 reproduction max diff=9.71e-17
Brier raw=0.1785 | sigmoid=0.1732 | isotonic=0.1757

R2_F4 (9/25)
Raw Step-7 reproduction max diff=9.71e-17
Brier raw=0.1632 | sigmoid=0.1579 | isotonic=0.1675

R2_F5 (10/25)
Raw Step-7 re

## 9. Calibration summary and decision table

In [11]:

summary = (
    fold_results
    .groupby(
        "calibration_method",
        as_index=False,
    )
    .agg(
        Brier=("brier_score", "mean"),
        Brier_SD=("brier_score", "std"),
        LogLoss=("log_loss", "mean"),
        LogLoss_SD=("log_loss", "std"),
        ECE=("ece_10bin", "mean"),
        Calibration_Intercept=(
            "calibration_intercept",
            "mean",
        ),
        Calibration_Slope=(
            "calibration_slope",
            "mean",
        ),
        Mean_Predicted_Probability=(
            "mean_predicted_probability",
            "mean",
        ),
        Observed_Adverse_Rate=(
            "observed_adverse_rate",
            "mean",
        ),
        ROC_AUC=("roc_auc", "mean"),
        PR_AUC=("pr_auc", "mean"),
        Recall=("recall_adverse", "mean"),
        Precision=("precision_adverse", "mean"),
        F1=("f1_adverse", "mean"),
        Balanced_Accuracy=(
            "balanced_accuracy",
            "mean",
        ),
        MCC=("mcc", "mean"),
    )
)

summary.to_csv(
    OUT_DIR
    / "german_calibration_summary.csv",
    index=False,
)

display(
    summary.sort_values(
        [
            "Brier",
            "LogLoss",
            "ECE",
        ],
        ascending=True,
    )
)


,calibration_method,Brier,Brier_SD,LogLoss,LogLoss_SD,ECE,Calibration_Intercept,Calibration_Slope,Mean_Predicted_Probability,Observed_Adverse_Rate,ROC_AUC,PR_AUC,Recall,Precision,F1,Balanced_Accuracy,MCC
1,Sigmoid,0.160564,0.011548,0.487191,0.030339,0.073567,0.031212,1.096170,0.302790,0.3,0.798802,0.634897,0.464135,0.646891,0.536208,0.677750,0.395607
0,Isotonic,0.162829,0.012362,0.501964,0.042407,0.066442,-0.082957,0.907247,0.302656,0.3,0.792547,0.596490,0.475006,0.643966,0.539162,0.679963,0.397587
2,Uncalibrated,0.165559,0.010302,0.498661,0.026643,0.091794,-0.396548,1.004470,0.363889,0.3,0.798802,0.634897,0.626506,0.586505,0.603075,0.718355,0.428590


## 10. Paired deltas against the uncalibrated hybrid

In [12]:

reference = (
    fold_results[
        fold_results[
            "calibration_method"
        ]
        == "Uncalibrated"
    ]
    [
        [
            "run_id",
            "brier_score",
            "log_loss",
            "ece_10bin",
            "roc_auc",
            "pr_auc",
            "recall_adverse",
            "f1_adverse",
            "balanced_accuracy",
            "mcc",
        ]
    ]
    .copy()
)

reference = reference.rename(
    columns={
        column: (
            "reference_"
            + column
        )
        for column
        in reference.columns
        if column != "run_id"
    }
)

paired = fold_results.merge(
    reference,
    on="run_id",
    how="left",
    validate="many_to_one",
)

delta_metrics = [
    "brier_score",
    "log_loss",
    "ece_10bin",
    "roc_auc",
    "pr_auc",
    "recall_adverse",
    "f1_adverse",
    "balanced_accuracy",
    "mcc",
]

for metric in delta_metrics:
    paired[
        "delta_"
        + metric
    ] = (
        paired[metric]
        - paired[
            "reference_"
            + metric
        ]
    )

paired.to_csv(
    OUT_DIR
    / "german_calibration_paired_deltas.csv",
    index=False,
)

delta_summary = (
    paired.groupby(
        "calibration_method",
        as_index=False,
    )
    .agg(
        Delta_Brier=(
            "delta_brier_score",
            "mean",
        ),
        Delta_LogLoss=(
            "delta_log_loss",
            "mean",
        ),
        Delta_ECE=(
            "delta_ece_10bin",
            "mean",
        ),
        Delta_ROC_AUC=(
            "delta_roc_auc",
            "mean",
        ),
        Delta_PR_AUC=(
            "delta_pr_auc",
            "mean",
        ),
        Delta_Recall=(
            "delta_recall_adverse",
            "mean",
        ),
        Delta_F1=(
            "delta_f1_adverse",
            "mean",
        ),
        Delta_Balanced_Accuracy=(
            "delta_balanced_accuracy",
            "mean",
        ),
        Delta_MCC=(
            "delta_mcc",
            "mean",
        ),
    )
)

delta_summary.to_csv(
    OUT_DIR
    / "german_calibration_delta_summary.csv",
    index=False,
)

display(delta_summary)


,calibration_method,Delta_Brier,Delta_LogLoss,Delta_ECE,Delta_ROC_AUC,Delta_PR_AUC,Delta_Recall,Delta_F1,Delta_Balanced_Accuracy,Delta_MCC
0,Isotonic,-0.002730,0.003304,-0.025351,-0.006255,-0.038407,-0.151500,-0.063913,-0.038392,-0.031003
1,Sigmoid,-0.004995,-0.011469,-0.018227,0.000000,0.000000,-0.162371,-0.066867,-0.040605,-0.032982
2,Uncalibrated,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


## 11. Descriptive calibration-curve data

In [13]:

def calibration_bins(
    y_true,
    probability,
    method_name,
    n_bins=10,
):
    work = pd.DataFrame({
        "y_true": np.asarray(
            y_true,
            dtype=int,
        ),
        "probability": np.asarray(
            probability,
            dtype=float,
        ),
    })

    # Quantile bins give more stable counts
    # for descriptive reliability diagrams.
    try:
        work["bin"] = pd.qcut(
            work["probability"],
            q=n_bins,
            duplicates="drop",
        )
    except Exception:
        work["bin"] = pd.cut(
            work["probability"],
            bins=n_bins,
            include_lowest=True,
        )

    curve = (
        work.groupby(
            "bin",
            observed=True,
        )
        .agg(
            mean_predicted_probability=(
                "probability",
                "mean",
            ),
            observed_adverse_rate=(
                "y_true",
                "mean",
            ),
            records=(
                "y_true",
                "size",
            ),
        )
        .reset_index(
            drop=True
        )
    )

    curve.insert(
        0,
        "calibration_method",
        method_name,
    )

    curve.insert(
        1,
        "bin_number",
        np.arange(
            1,
            len(curve) + 1,
        ),
    )

    return curve


curve_tables = []

for method_name, probability_column in {
    "Uncalibrated": "uncalibrated_probability",
    "Sigmoid": "sigmoid_probability",
    "Isotonic": "isotonic_probability",
}.items():
    curve_tables.append(
        calibration_bins(
            predictions["y_true"],
            predictions[
                probability_column
            ],
            method_name,
            n_bins=10,
        )
    )

calibration_curve_data = pd.concat(
    curve_tables,
    ignore_index=True,
)

calibration_curve_data.to_csv(
    OUT_DIR
    / "german_calibration_curve_data.csv",
    index=False,
)

display(calibration_curve_data)


,calibration_method,bin_number,mean_predicted_probability,observed_adverse_rate,records
0,Uncalibrated,1,0.040575,0.034000,500
1,Uncalibrated,2,0.087335,0.078000,500
2,Uncalibrated,3,0.142171,0.106000,500
3,Uncalibrated,4,0.207256,0.182000,500
4,Uncalibrated,5,0.279884,0.180000,500
5,Uncalibrated,6,0.371300,0.272000,500
6,Uncalibrated,7,0.469426,0.346000,500
7,Uncalibrated,8,0.566740,0.474000,500
8,Uncalibrated,9,0.675004,0.586000,500
9,Uncalibrated,10,0.799202,0.742000,500


## 12. Final consistency checks

Do not choose the calibration method manually from one fold.

After running this notebook, upload the entire Step-9 output folder. The final calibration method will be frozen after reviewing the **German aggregate results only**, and then independently replicated on Australian and Taiwan.


In [14]:

assert len(
    fold_results
) == (
    25
    * len(
        CALIBRATION_METHODS
    )
)

assert len(
    predictions
) == 5000

assert len(
    calibrators
) == 25

assert len(
    inner_oof
) == (
    25
    * 800
)

for probability_column in [
    "uncalibrated_probability",
    "sigmoid_probability",
    "isotonic_probability",
]:
    assert predictions[
        probability_column
    ].between(
        0.0,
        1.0,
    ).all()

assert (
    fold_results[
        "brier_score"
    ] >= 0
).all()

assert (
    fold_results[
        "log_loss"
    ] >= 0
).all()

configuration = {
    "stage": (
        "Objective 3 Step 9 - "
        "German probability calibration development"
    ),
    "development_dataset": (
        "German Credit"
    ),
    "frozen_core_hybrid": {
        "feature_selection": (
            "Group-aware Chi-Square Top75"
        ),
        "base_models": [
            "Balanced LR",
            "Balanced RF",
            "Balanced XGB",
        ],
        "fusion": (
            "Equal-probability soft voting"
        ),
    },
    "calibration_candidates": (
        CALIBRATION_METHODS
    ),
    "calibration_fit_data": (
        "Inner 5-fold grouped OOF hybrid probabilities "
        "from each outer-training partition"
    ),
    "outer_validation": (
        "Same 5-fold x 5-repeat grouped folds "
        "used in previous steps"
    ),
    "primary_selection_criteria": [
        "Brier score",
        "Log loss",
        "ECE",
        "Calibration intercept",
        "Calibration slope",
    ],
    "calibration_selection_rule": (
        "Method will be frozen using German evidence only; "
        "Australian and Taiwan cannot influence method selection."
    ),
    "threshold_optimization": (
        "Deferred to a later step"
    ),
}

with open(
    OUT_DIR
    / "step9_experiment_configuration.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        configuration,
        f,
        indent=4,
    )

manifest = sorted(
    [
        p.name
        for p
        in OUT_DIR.iterdir()
        if p.is_file()
    ]
)

pd.DataFrame(
    {
        "generated_file": manifest
    }
).to_csv(
    OUT_DIR
    / "step9_output_manifest.csv",
    index=False,
)

print("=" * 80)
print("STEP 9 COMPLETED SUCCESSFULLY")
print("=" * 80)
print("Output folder:", OUT_DIR)
print("\nMost important files:")
print(" - german_calibration_summary.csv")
print(" - german_calibration_delta_summary.csv")
print(" - german_calibration_fold_results.csv")
print(" - german_calibration_outer_predictions.csv")
print(" - german_calibration_curve_data.csv")
print(" - german_calibrator_parameters.csv")


STEP 9 COMPLETED SUCCESSFULLY
Output folder: D:\PHD\Research Paper writing\3rd Obj. paper\results\calibration_german_development

Most important files:
 - german_calibration_summary.csv
 - german_calibration_delta_summary.csv
 - german_calibration_fold_results.csv
 - german_calibration_outer_predictions.csv
 - german_calibration_curve_data.csv
 - german_calibrator_parameters.csv
